# 과제2(정합성 검증 미니 엔진) + 과제3(자연어 모듈 조합) 취합

### 생성 결과

- `equipment_data.json`
- `equipment_data.csv`
- `line_data.json`
- `line_data.csv`
- `rules.json`
- `task2_readme_rules.json`
- `unresolved_line_references.csv`
- `violation_report.csv`

## 0. 파일 경로 설정

In [1]:
import pandas as pd
from pathlib import Path
import json, re
from typing import Any, Dict, List, Optional


# 같은 폴더에 파일을 두고 실행하는 경우
EXCEL_PATH = Path("consistency_check.xlsx")
MD_PATH = Path("task3_nl_to_modules.md")

# ChatGPT/Colab 환경에서 실행하는 경우 fallback
if not EXCEL_PATH.exists():
    EXCEL_PATH = Path("/mnt/data/consistency_check.xlsx")

if not MD_PATH.exists():
    MD_PATH = Path("/mnt/data/task3_nl_to_modules.md")

OUT_DIR = Path("equipment_rules_output")
OUT_DIR.mkdir(exist_ok=True)

print("Excel 파일:", EXCEL_PATH)
print("MD 파일:", MD_PATH)
print("결과 저장 폴더:", OUT_DIR.resolve())

Excel 파일: consistency_check.xlsx
MD 파일: task3_nl_to_modules.md
결과 저장 폴더: C:\Users\Administrator\Desktop\technical_review\equipment_rules_output


## 1. 공통 유틸 함수

In [2]:
def clean_value(value):
    """Excel 공란 NaN을 None으로 변환합니다."""
    if pd.isna(value):
        return None
    return value


def to_float_or_none(value):
    """숫자로 변환 가능한 값은 float, 불가능하면 None을 반환합니다."""
    value = clean_value(value)

    if value is None or value == "":
        return None

    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def yes_no_to_bool(value):
    """Y/N, Yes/No 값을 bool로 변환합니다."""
    value = clean_value(value)

    if value is None:
        return None

    text = str(value).strip().upper()

    if text in {"Y", "YES", "TRUE", "1"}:
        return True

    if text in {"N", "NO", "FALSE", "0"}:
        return False

    return None


def normalize_class_name(raw_type):
    """Equipment Type을 과제3 ClassName으로 표준화합니다."""
    text = str(clean_value(raw_type) or "").strip().lower()

    if "tank" in text:
        return "Tank"

    if "pump" in text:
        return "Pump"

    if "heat" in text and "exchanger" in text:
        return "HeatExchanger"

    if "vessel" in text:
        return "Vessel"

    if "compressor" in text:
        return "Compressor"

    return "Equipment"


def read_excel_safely(excel_path):
    """Excel의 모든 시트를 읽습니다."""
    return pd.read_excel(excel_path, sheet_name=None)

## 2. Excel 시트 구조 확인

In [3]:
sheets = read_excel_safely(EXCEL_PATH)

print("시트 목록:", list(sheets.keys()))

for sheet_name, df in sheets.items():
    print("\n===", sheet_name, "===")
    print("shape:", df.shape)
    print("columns:", list(df.columns))
    display(df.head())

시트 목록: ['Equipment_List', 'Line_List', 'READ_ME']

=== Equipment_List ===
shape: (10, 8)
columns: ['Equipment Tag', 'Type', 'Operating Pressure (barg)', 'Design Pressure (barg)', 'Operating Temp (degC)', 'Design Temp (degC)', 'Material', 'Has Safety Valve']


,Equipment Tag,Type,Operating Pressure (barg),Design Pressure (barg),Operating Temp (degC),Design Temp (degC),Material,Has Safety Valve
0,T-101,Storage Tank,1,1.05,40,70,CS,N
1,P-101A,Centrifugal Pump,5,6.00,40,90,CS,N
2,P-101B,Centrifugal Pump,5,5.20,40,90,CS,N
3,E-101,Heat Exchanger,8,10.00,120,160,SS304,Y
4,E-102,Heat Exchanger,6,8.00,110,150,SS304,N



=== Line_List ===
shape: (8, 6)
columns: ['배관 번호', 'From 장비', 'To 장비', '설계압력 [kPag]', '운전온도 [K]', '보온 여부']


,배관 번호,From 장비,To 장비,설계압력 [kPag],운전온도 [K],보온 여부
0,L-001,T-101,P-101A,105,313,N
1,L-002,T-101,P-101B,105,313,N
2,L-010,P-101A,E-101,600,313,Y
3,L-011,P-101C,E-101,600,313,Y
4,L-020,E-101,V-201,800,393,Y



=== READ_ME ===
shape: (17, 1)
columns: ['READ ME — 과제 2 입력 데이터']


,READ ME — 과제 2 입력 데이터
0,제출 기한: 2026년 6월 30일(화)까지.
1,NaN
2,이 워크북에는 두 개의 데이터 시트가 있습니다: 'Equipment_List'(장비...
3,"서로 다른 엔지니어가 만들었다는 설정이라, 같은 물리적 속성이 시트마다 다른 이름·..."
4,"(예: 영문 표기 vs 한글 표기, barg vs kPag, ℃ vs K)"


## 3. Equipment_List + Line_List → EQUIPMENT_DATA 생성

In [4]:
def build_equipment_data(excel_path):
    """
    Equipment_List와 Line_List를 읽어
    EQUIPMENT_DATA, LINE_DATA, unresolved_references를 생성합니다.
    """
    sheets = read_excel_safely(excel_path)

    eq_raw = sheets["Equipment_List"].copy()
    line_raw = sheets["Line_List"].copy()

    equipment_data = []

    for _, row in eq_raw.iterrows():
        tag = clean_value(row.get("Equipment Tag"))

        if tag is None:
            continue

        item = {
            "Tag": str(tag).strip(),
            "OriginalType": clean_value(row.get("Type")),
            "ClassName": normalize_class_name(row.get("Type")),
            "OperatingPressure": to_float_or_none(row.get("Operating Pressure (barg)")),
            "DesignPressure": to_float_or_none(row.get("Design Pressure (barg)")),
            "OperatingTemperature": to_float_or_none(row.get("Operating Temp (degC)")),
            "DesignTemperature": to_float_or_none(row.get("Design Temp (degC)")),
            "Material": clean_value(row.get("Material")),
            "HasSafetyValve": yes_no_to_bool(row.get("Has Safety Valve")),
            "IncomingLines": [],
            "OutgoingLines": [],
            "ConnectedLines": [],
        }

        equipment_data.append(item)

    equipment_by_tag = {item["Tag"]: item for item in equipment_data}

    line_data = []
    unresolved_references = []

    for _, row in line_raw.iterrows():
        line_no = clean_value(row.get("배관 번호"))

        if line_no is None:
            continue

        from_tag = clean_value(row.get("From 장비"))
        to_tag = clean_value(row.get("To 장비"))

        # kPag → barg
        design_pressure_kpag = to_float_or_none(row.get("설계압력 [kPag]"))
        design_pressure_barg = None if design_pressure_kpag is None else design_pressure_kpag / 100.0

        # K → degC
        operating_temp_k = to_float_or_none(row.get("운전온도 [K]"))
        operating_temp_deg_c = None if operating_temp_k is None else operating_temp_k - 273.15

        line_item = {
            "LineNo": str(line_no).strip(),
            "FromTag": None if from_tag is None else str(from_tag).strip(),
            "ToTag": None if to_tag is None else str(to_tag).strip(),
            "DesignPressure": design_pressure_barg,
            "DesignPressureOriginal": design_pressure_kpag,
            "DesignPressureOriginalUnit": "kPag",
            "OperatingTemperature": operating_temp_deg_c,
            "OperatingTemperatureOriginal": operating_temp_k,
            "OperatingTemperatureOriginalUnit": "K",
            "Insulated": yes_no_to_bool(row.get("보온 여부")),
        }

        line_data.append(line_item)

        from_clean = line_item["FromTag"]
        to_clean = line_item["ToTag"]

        if from_clean in equipment_by_tag:
            equipment_by_tag[from_clean]["OutgoingLines"].append(line_item["LineNo"])
            equipment_by_tag[from_clean]["ConnectedLines"].append(line_item["LineNo"])
        else:
            unresolved_references.append({
                "LineNo": line_item["LineNo"],
                "Column": "FromTag",
                "MissingTag": from_clean,
                "Reason": "Line_List의 From 장비가 Equipment_List에 없음",
            })

        if to_clean in equipment_by_tag:
            equipment_by_tag[to_clean]["IncomingLines"].append(line_item["LineNo"])
            equipment_by_tag[to_clean]["ConnectedLines"].append(line_item["LineNo"])
        else:
            unresolved_references.append({
                "LineNo": line_item["LineNo"],
                "Column": "ToTag",
                "MissingTag": to_clean,
                "Reason": "Line_List의 To 장비가 Equipment_List에 없음",
            })

    return equipment_data, line_data, unresolved_references


equipment_data, line_data, unresolved_references = build_equipment_data(EXCEL_PATH)

print("equipment_data 개수:", len(equipment_data))
print("line_data 개수:", len(line_data))
print("unresolved_references 개수:", len(unresolved_references))

display(pd.DataFrame(equipment_data))
display(pd.DataFrame(line_data))
display(pd.DataFrame(unresolved_references))

equipment_data 개수: 10
line_data 개수: 8
unresolved_references 개수: 1


,Tag,OriginalType,ClassName,OperatingPressure,DesignPressure,OperatingTemperature,DesignTemperature,Material,HasSafetyValve,IncomingLines,OutgoingLines,ConnectedLines
0,T-101,Storage Tank,Tank,1.0,1.05,40.0,70.0,CS,False,[],"[L-001, L-002]","[L-001, L-002]"
1,P-101A,Centrifugal Pump,Pump,5.0,6.00,40.0,90.0,CS,False,[L-001],[L-010],"[L-001, L-010]"
2,P-101B,Centrifugal Pump,Pump,5.0,5.20,40.0,90.0,CS,False,[L-002],[],[L-002]
3,E-101,Heat Exchanger,HeatExchanger,8.0,10.00,120.0,160.0,SS304,True,"[L-010, L-011]",[L-020],"[L-010, L-011, L-020]"
4,E-102,Heat Exchanger,HeatExchanger,6.0,8.00,110.0,150.0,SS304,False,[],[],[]
5,V-201,Pressure Vessel,Vessel,12.0,15.00,150.0,190.0,SS316,True,[L-020],[L-021],"[L-020, L-021]"
6,V-202,Pressure Vessel,Vessel,12.0,NaN,150.0,190.0,SS316,True,"[L-030, L-040]",[],"[L-030, L-040]"
7,C-301,Compressor,Compressor,20.0,25.00,200.0,215.0,CS,True,[L-021],[L-030],"[L-021, L-030]"
8,PUMP102,Centrifugal Pump,Pump,4.0,5.00,40.0,90.0,CS,False,[],[],[]
9,P-103,Centrifugal Pump,Pump,3.0,4.00,40.0,90.0,CS,False,[],[L-040],[L-040]


,LineNo,FromTag,ToTag,DesignPressure,DesignPressureOriginal,DesignPressureOriginalUnit,OperatingTemperature,OperatingTemperatureOriginal,OperatingTemperatureOriginalUnit,Insulated
0,L-001,T-101,P-101A,1.05,105.0,kPag,39.85,313.0,K,False
1,L-002,T-101,P-101B,1.05,105.0,kPag,39.85,313.0,K,False
2,L-010,P-101A,E-101,6.00,600.0,kPag,39.85,313.0,K,True
3,L-011,P-101C,E-101,6.00,600.0,kPag,39.85,313.0,K,True
4,L-020,E-101,V-201,8.00,800.0,kPag,119.85,393.0,K,True
5,L-021,V-201,C-301,15.00,1500.0,kPag,149.85,423.0,K,True
6,L-030,C-301,V-202,25.00,2500.0,kPag,199.85,473.0,K,True
7,L-040,P-103,V-202,4.00,400.0,kPag,39.85,313.0,K,False


,LineNo,Column,MissingTag,Reason
0,L-011,FromTag,P-101C,Line_List의 From 장비가 Equipment_List에 없음


## 4. md 파일에서 과제3 자연어 문장 추출

In [5]:
def extract_task3_sentences_from_md(md_path):
    """task3 md 파일에서 변환할 문장 12개를 추출합니다."""
    text = md_path.read_text(encoding="utf-8")

    sentences = []
    current = None

    for raw_line in text.splitlines():
        line = raw_line.strip()
        match = re.match(r"^(\d{1,2})\.\s+(.*)", line)

        if match:
            if current:
                sentences.append(current.strip())

            current = match.group(2).strip()
            continue

        if current and line and not line.startswith(">") and not line.startswith("#") and not line.startswith("```"):
            if not line.startswith("-") and "일부 문장은" not in line and "여기서" not in line:
                current += " " + line

    if current:
        sentences.append(current.strip())

    return sentences[:12]


sentences = extract_task3_sentences_from_md(MD_PATH)

print("추출된 문장 수:", len(sentences))

for i, sentence in enumerate(sentences, start=1):
    print(f"{i}. {sentence}")

추출된 문장 수: 12
1. 모든 용기(vessel)의 설계압력은 운전압력의 1.1배 이상이어야 한다.
2. 안전밸브가 있는 모든 장비는 설계압력이 정의되어 있어야 한다.
3. 모든 열교환기의 설계온도는 운전온도보다 30 이상 높아야 한다.
4. 모든 장비 태그는 정해진 태그 패턴과 일치해야 한다.
5. 모든 펌프는 재질과 설계압력이 모두 정의되어 있어야 한다.
6. 어떤 탱크도 운전압력이 2 barg를 초과해서는 안 된다.
7. 운전온도가 200 이상인 모든 용기는 설계온도가 240 이상이어야 한다.
8. 안전밸브가 있는 모든 장비는 설계압력이 운전압력의 1.2배 이상이어야 한다.
9. 모든 압축기는 설계압력이 운전압력의 1.1배 이상이고, 동시에 설계온도가 운전온도보다 30 이상 높아야 한다.
10. 모든 열교환기의 재질은 SS304 또는 SS316 중 하나여야 한다.
11. 어떤 장비도 설계온도가 운전온도보다 낮아서는 안 된다.
12. 모든 펌프 태그는 태그 패턴과 일치해야 하고, 설계압력이 정의되어 있어야 한다. 변환기가 조합식을 만들어낸 뒤, 그 식이 위 모듈 명세에 맞는 올바른 형태인지 검사하는 **결정론적 검증기(validator)** 를 돌리세요. 식이 검증을 통과하지 못하면, **구체적인 실패 사유를 피드백해서 다시 생성**하세요 — 잘못된 식을 그대로 받아들이지 마세요. 문장별로 재생성이 몇 번 필요했고 어떤 실패가 났는지 기록하세요.


## 5. 자연어 문장 → RULES용 AST 생성

In [6]:
def sentence_to_rule_ast(sentence, rule_id):
    """
    과제3의 12개 문장을 AST로 변환합니다.
    여기서는 LLM이 아니라 문장 패턴 기반 결정론적 매핑을 사용합니다.
    """
    s = sentence.strip()

    if "모든 용기" in s and "1.1배" in s and "설계압력" in s:
        ast_expr = ["forEvery", ["allOf", "Vessel"], ["isAtLeast", ["prop", "x", "DesignPressure"], ["times", ["prop", "x", "OperatingPressure"], 1.1]]]

    elif "안전밸브가 있는 모든 장비" in s and "정의" in s and "1.2배" not in s:
        ast_expr = ["forEvery", ["withSafetyValve", "Equipment"], ["exists", ["prop", "x", "DesignPressure"]]]

    elif "모든 열교환기" in s and "설계온도" in s and "30" in s:
        ast_expr = ["forEvery", ["allOf", "HeatExchanger"], ["isAtLeast", ["prop", "x", "DesignTemperature"], ["plus", ["prop", "x", "OperatingTemperature"], 30]]]

    elif "모든 장비 태그" in s:
        ast_expr = ["forEvery", ["allOf", "Equipment"], ["matchesPattern", ["prop", "x", "Tag"], "TAG_PATTERN"]]

    elif "모든 펌프" in s and "재질" in s and "설계압력" in s and "정의" in s:
        ast_expr = ["forEvery", ["allOf", "Pump"], ["and", ["exists", ["prop", "x", "Material"]], ["exists", ["prop", "x", "DesignPressure"]]]]

    elif "어떤 탱크도" in s and "2 barg" in s:
        ast_expr = ["forEvery", ["allOf", "Tank"], ["isAtMost", ["prop", "x", "OperatingPressure"], 2]]

    elif "운전온도가 200 이상" in s and "설계온도가 240 이상" in s:
        ast_expr = ["forEvery", ["allOf", "Vessel"], ["or", ["not", ["isAtLeast", ["prop", "x", "OperatingTemperature"], 200]], ["isAtLeast", ["prop", "x", "DesignTemperature"], 240]]]

    elif "안전밸브가 있는 모든 장비" in s and "1.2배" in s:
        ast_expr = ["forEvery", ["withSafetyValve", "Equipment"], ["isAtLeast", ["prop", "x", "DesignPressure"], ["times", ["prop", "x", "OperatingPressure"], 1.2]]]

    elif "모든 압축기" in s and "1.1배" in s and "30" in s:
        ast_expr = ["forEvery", ["allOf", "Compressor"], ["and", ["isAtLeast", ["prop", "x", "DesignPressure"], ["times", ["prop", "x", "OperatingPressure"], 1.1]], ["isAtLeast", ["prop", "x", "DesignTemperature"], ["plus", ["prop", "x", "OperatingTemperature"], 30]]]]

    elif "모든 열교환기의 재질" in s and "SS304" in s and "SS316" in s:
        ast_expr = ["forEvery", ["allOf", "HeatExchanger"], ["or", ["isEqual", ["prop", "x", "Material"], "SS304"], ["isEqual", ["prop", "x", "Material"], "SS316"]]]

    elif "어떤 장비도" in s and "설계온도가 운전온도보다 낮아서는 안" in s:
        ast_expr = ["forEvery", ["allOf", "Equipment"], ["isAtLeast", ["prop", "x", "DesignTemperature"], ["prop", "x", "OperatingTemperature"]]]

    elif "모든 펌프 태그" in s and "태그 패턴" in s:
        ast_expr = ["forEvery", ["allOf", "Pump"], ["and", ["matchesPattern", ["prop", "x", "Tag"], "TAG_PATTERN"], ["exists", ["prop", "x", "DesignPressure"]]]]

    else:
        raise ValueError(f"지원하지 않는 문장 패턴입니다: {sentence}")

    return {
        "rule_id": rule_id,
        "source_sentence": sentence,
        "ast": ast_expr,
    }


rules = [sentence_to_rule_ast(sentence, f"T3_{i:02d}") for i, sentence in enumerate(sentences, start=1)]

print("rules 개수:", len(rules))
display(pd.DataFrame(rules))

rules 개수: 12


,rule_id,source_sentence,ast
0,T3_01,모든 용기(vessel)의 설계압력은 운전압력의 1.1배 이상이어야 한다.,"[forEvery, [allOf, Vessel], [isAtLeast, [prop,..."
1,T3_02,안전밸브가 있는 모든 장비는 설계압력이 정의되어 있어야 한다.,"[forEvery, [withSafetyValve, Equipment], [exis..."
2,T3_03,모든 열교환기의 설계온도는 운전온도보다 30 이상 높아야 한다.,"[forEvery, [allOf, HeatExchanger], [isAtLeast,..."
3,T3_04,모든 장비 태그는 정해진 태그 패턴과 일치해야 한다.,"[forEvery, [allOf, Equipment], [matchesPattern..."
4,T3_05,모든 펌프는 재질과 설계압력이 모두 정의되어 있어야 한다.,"[forEvery, [allOf, Pump], [and, [exists, ['pro..."
5,T3_06,어떤 탱크도 운전압력이 2 barg를 초과해서는 안 된다.,"[forEvery, [allOf, Tank], [isAtMost, [prop, x,..."
6,T3_07,운전온도가 200 이상인 모든 용기는 설계온도가 240 이상이어야 한다.,"[forEvery, [allOf, Vessel], [or, [not, ['isAtL..."
7,T3_08,안전밸브가 있는 모든 장비는 설계압력이 운전압력의 1.2배 이상이어야 한다.,"[forEvery, [withSafetyValve, Equipment], [isAt..."
8,T3_09,"모든 압축기는 설계압력이 운전압력의 1.1배 이상이고, 동시에 설계온도가 운전온도보...","[forEvery, [allOf, Compressor], [and, [isAtLea..."
9,T3_10,모든 열교환기의 재질은 SS304 또는 SS316 중 하나여야 한다.,"[forEvery, [allOf, HeatExchanger], [or, [isEqu..."


## 6. fallback: Excel READ_ME 시트에서 과제2 규칙 추출

In [7]:
def extract_task2_rules_from_excel_readme(excel_path):
    """
    Excel READ_ME 시트에서 R1~R5 규칙을 추출합니다.
    md 파일을 못 쓰는 경우를 대비한 fallback입니다.
    """
    sheets = read_excel_safely(excel_path)

    if "READ_ME" not in sheets:
        return []

    readme_df = sheets["READ_ME"]
    lines = [str(v).strip() for v in readme_df.iloc[:, 0].dropna().tolist()]
    rule_lines = [line for line in lines if re.match(r"^R\d+\.", line)]

    task2_rules = []

    for line in rule_lines:
        rule_no = line.split(".", 1)[0]
        sentence = line.split(".", 1)[1].strip()

        ast_expr = None

        if rule_no == "R1":
            ast_expr = ["forEvery", ["allOf", "Equipment"], ["isAtLeast", ["prop", "x", "DesignPressure"], ["times", ["prop", "x", "OperatingPressure"], 1.1]]]

        elif rule_no == "R3":
            ast_expr = ["forEvery", ["allOf", "Equipment"], ["isAtLeast", ["prop", "x", "DesignTemperature"], ["plus", ["prop", "x", "OperatingTemperature"], 30]]]

        elif rule_no == "R4":
            ast_expr = ["forEvery", ["withSafetyValve", "Equipment"], ["exists", ["prop", "x", "DesignPressure"]]]

        elif rule_no == "R5":
            ast_expr = ["forEvery", ["allOf", "Equipment"], ["matchesPattern", ["prop", "x", "Tag"], "TAG_PATTERN"]]

        task2_rules.append({
            "rule_id": rule_no,
            "source_sentence": sentence,
            "ast": ast_expr,
            "note": "R2는 Line_List와 Equipment_List 간 참조 무결성 검사이므로 별도 cross-sheet 검사 필요" if rule_no == "R2" else None,
        })

    return task2_rules


task2_readme_rules = extract_task2_rules_from_excel_readme(EXCEL_PATH)

print("READ_ME에서 추출한 규칙 수:", len(task2_readme_rules))
display(pd.DataFrame(task2_readme_rules))

READ_ME에서 추출한 규칙 수: 5


,rule_id,source_sentence,ast,note
0,R1,모든 장비의 설계압력은 운전압력의 1.1배 이상이어야 한다.,"[forEvery, [allOf, Equipment], [isAtLeast, [pr...",None
1,R2,배관 목록(From/To)에서 참조하는 모든 장비 태그는 장비 목록에 존재해야 한다.,None,R2는 Line_List와 Equipment_List 간 참조 무결성 검사이므로 별...
2,R3,모든 장비의 설계온도는 운전온도보다 30℃ 이상 높아야 한다(설계온도 ≥ 운전온도 ...,"[forEvery, [allOf, Equipment], [isAtLeast, [pr...",None
3,R4,안전밸브가 있는 장비는 설계압력이 반드시 정의(공란이 아님)되어 있어야 한다.,"[forEvery, [withSafetyValve, Equipment], [exis...",None
4,R5,모든 장비 태그는 다음 패턴을 따라야 한다: 영문자 1자 이상 + 하이픈 + 숫자 ...,"[forEvery, [allOf, Equipment], [matchesPattern...",None


## 7. AST를 EQUIPMENT_DATA에 적용하는 실행기

In [8]:
def list_to_tuple(obj):
    """list 기반 AST를 tuple 기반 AST로 바꿉니다."""
    if isinstance(obj, list):
        return tuple(list_to_tuple(x) for x in obj)
    return obj


def select_targets(selector, data):
    """selector AST를 실제 데이터 필터링으로 실행합니다."""
    selector = list_to_tuple(selector)
    func = selector[0]

    if func == "allOf":
        class_name = selector[1]

        if class_name == "Equipment":
            return data

        return [row for row in data if row.get("ClassName") == class_name]

    if func == "withSafetyValve":
        class_name = selector[1]
        rows = [row for row in data if row.get("HasSafetyValve") is True]

        if class_name == "Equipment":
            return rows

        return [row for row in rows if row.get("ClassName") == class_name]

    raise ValueError(f"지원하지 않는 selector 함수: {func}")


def eval_value(expr, x):
    """value AST를 실제 값으로 계산합니다."""
    expr = list_to_tuple(expr)

    if isinstance(expr, (int, float, str)) and expr != "x":
        return expr

    func = expr[0]

    if func == "prop":
        _, target, property_name = expr

        if target != "x":
            raise ValueError("prop의 첫 번째 인자는 x여야 합니다.")

        return x.get(property_name)

    if func == "times":
        _, a, k = expr
        value = eval_value(a, x)
        return None if value is None else value * k

    if func == "plus":
        _, a, k = expr
        value = eval_value(a, x)
        return None if value is None else value + k

    raise ValueError(f"지원하지 않는 value 함수: {func}")


def matches_tag_pattern(tag):
    """태그 패턴 검사: 예 P-101, E-102, V-201, P-101A"""
    if tag is None:
        return False

    return re.match(r"^[A-Za-z]+-\d{3}[A-Za-z]?$", str(tag)) is not None


def eval_condition(expr, x):
    """condition AST를 True/False로 평가합니다."""
    expr = list_to_tuple(expr)
    func = expr[0]

    if func == "exists":
        value = eval_value(expr[1], x)
        return value is not None and value != ""

    if func == "isAtLeast":
        left = eval_value(expr[1], x)
        right = eval_value(expr[2], x)

        if left is None or right is None:
            return False

        return left >= right

    if func == "isAtMost":
        left = eval_value(expr[1], x)
        right = eval_value(expr[2], x)

        if left is None or right is None:
            return False

        return left <= right

    if func == "isEqual":
        left = eval_value(expr[1], x)
        right = eval_value(expr[2], x)
        return left == right

    if func == "matchesPattern":
        value = eval_value(expr[1], x)
        regex_name = expr[2]

        if regex_name == "TAG_PATTERN":
            return matches_tag_pattern(value)

        raise ValueError(f"지원하지 않는 regexName: {regex_name}")

    if func == "and":
        return all(eval_condition(child, x) for child in expr[1:])

    if func == "or":
        return any(eval_condition(child, x) for child in expr[1:])

    if func == "not":
        return not eval_condition(expr[1], x)

    raise ValueError(f"지원하지 않는 condition 함수: {func}")


def evaluate_rule(rule, equipment_data):
    """규칙 하나를 equipment_data 전체에 적용합니다."""
    ast_expr = rule.get("ast")

    if ast_expr is None:
        return []

    ast_expr = list_to_tuple(ast_expr)

    if ast_expr[0] != "forEvery":
        raise ValueError("최상위 함수는 forEvery여야 합니다.")

    _, selector, condition = ast_expr

    targets = select_targets(selector, equipment_data)
    violations = []

    for row in targets:
        passed = eval_condition(condition, row)

        if not passed:
            violations.append({
                "rule_id": rule["rule_id"],
                "source_sentence": rule["source_sentence"],
                "tag": row.get("Tag"),
                "class_name": row.get("ClassName"),
                "reason": "조건을 만족하지 않음",
            })

    return violations


def build_violation_report(equipment_data, rules):
    """여러 규칙을 적용해 전체 위반 리포트를 생성합니다."""
    all_violations = []

    for rule in rules:
        all_violations.extend(evaluate_rule(rule, equipment_data))

    return all_violations


violation_report = build_violation_report(equipment_data, rules)

print("위반 건수:", len(violation_report))
display(pd.DataFrame(violation_report))

위반 건수: 6


,rule_id,source_sentence,tag,class_name,reason
0,T3_01,모든 용기(vessel)의 설계압력은 운전압력의 1.1배 이상이어야 한다.,V-202,Vessel,조건을 만족하지 않음
1,T3_02,안전밸브가 있는 모든 장비는 설계압력이 정의되어 있어야 한다.,V-202,Vessel,조건을 만족하지 않음
2,T3_04,모든 장비 태그는 정해진 태그 패턴과 일치해야 한다.,PUMP102,Pump,조건을 만족하지 않음
3,T3_08,안전밸브가 있는 모든 장비는 설계압력이 운전압력의 1.2배 이상이어야 한다.,V-202,Vessel,조건을 만족하지 않음
4,T3_09,"모든 압축기는 설계압력이 운전압력의 1.1배 이상이고, 동시에 설계온도가 운전온도보...",C-301,Compressor,조건을 만족하지 않음
5,T3_12,"모든 펌프 태그는 태그 패턴과 일치해야 하고, 설계압력이 정의되어 있어야 한다. 변...",PUMP102,Pump,조건을 만족하지 않음


## 8. 결과 저장

In [9]:
def save_json(path, data):
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")


save_json(OUT_DIR / "equipment_data.json", equipment_data)
save_json(OUT_DIR / "line_data.json", line_data)
save_json(OUT_DIR / "rules.json", rules)
save_json(OUT_DIR / "task2_readme_rules.json", task2_readme_rules)

pd.DataFrame(equipment_data).to_csv(OUT_DIR / "equipment_data.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(line_data).to_csv(OUT_DIR / "line_data.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(unresolved_references).to_csv(OUT_DIR / "unresolved_line_references.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(violation_report).to_csv(OUT_DIR / "violation_report.csv", index=False, encoding="utf-8-sig")

print("저장 완료:", OUT_DIR.resolve())

for p in OUT_DIR.iterdir():
    print("-", p.name)

저장 완료: C:\Users\Administrator\Desktop\technical_review\equipment_rules_output
- equipment_data.csv
- equipment_data.json
- line_data.csv
- line_data.json
- rules.json
- task2_readme_rules.json
- unresolved_line_references.csv
- violation_report.csv
